# Leakage-safe day-ahead electricity load forecasting

This notebook is a presentation layer over the tested `load_forecasting` package. It reproduces the frozen 24-hour-ahead experiment without duplicating research logic in notebook cells.

**Interpretation boundary:** Victoria demand is real, but target-time temperature is observed rather than an archived forecast. Scores are retrospective oracle-weather benchmarks, not deployment claims.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from load_forecasting.data import load_hourly_data  # noqa: E402
from load_forecasting.experiment import ExperimentConfig, run_experiment, save_result  # noqa: E402

## 1. Inspect the series

The loader enforces unique, complete hourly timestamps and finite, positive measurements before any modeling begins.

In [ ]:
data_path = PROJECT_ROOT / "data" / "victoria_hourly_load.csv"
data = load_hourly_data(data_path)
display(data.describe(include="all"))

figure, axis = plt.subplots(figsize=(13, 4))
axis.plot(data["datetime"].iloc[: 24 * 28], data["load_mw"].iloc[: 24 * 28], linewidth=0.9)
axis.set(title="First four weeks of Victoria operational demand", xlabel="Time", ylabel="Load (MW)")
axis.grid(alpha=0.2)
figure.autofmt_xdate()
plt.show()

## 2. Run the frozen protocol

For a target at time $t$, the issue time is $t-24$ hours. Four expanding-window folds select the model, a later block calibrates the interval, and the final 20% remains an untouched test set.

In [ ]:
config = ExperimentConfig(horizon_hours=24, cv_splits=4, random_state=42)
result = run_experiment(data_path, config)
save_result(result, PROJECT_ROOT / "artifacts" / "victoria", PROJECT_ROOT / "figures" / "victoria")
result.metrics.style.format({column: "{:.3f}" for column in result.metrics if column != "model"})

In [ ]:
cv_summary = result.cv_results.groupby("model")["rmse_mw"].agg(["mean", "std"]).sort_values("mean")
display(cv_summary.style.format("{:.2f}"))
display(pd.Series(result.interval_summary, name="value").to_frame())
display(pd.Series(result.dm_test, name="value").to_frame())

## 3. Diagnose, do not just rank

The selected model must be read alongside interval calibration, signed bias, residual structure, subgroup error, and data limitations. Permutation importance is predictive—not causal.

In [ ]:
display(Image(filename=PROJECT_ROOT / "figures" / "victoria" / "holdout_forecast.png"))
display(Image(filename=PROJECT_ROOT / "figures" / "victoria" / "residual_diagnostics.png"))
display(Image(filename=PROJECT_ROOT / "figures" / "victoria" / "feature_importance.png"))

## 4. Conclusion and next test

Histogram gradient boosting wins the declared validation rule and improves materially over weekly seasonal naive. However, the 90% interval is very conservative and a large June residual episode remains. The scientifically useful next step is a frozen replication using archived weather forecasts and a new external holdout—not post-hoc tuning here.